# LightGBM — A Highly Efficient Gradient Boosting Decision Tree (2017)

_Papers · Classical ML_

**Train gradient-boosted trees up to ~20x faster by sampling the data smartly (GOSS) and bundling sparse features (EFB) — at almost the same accuracy.**

---

This notebook is a practice scaffold for the **LightGBM — A Highly Efficient Gradient Boosting Decision Tree (2017)** lesson. Run the cells, then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
import numpy as np, matplotlib.pyplot as plt

## Reference implementation — LightGBM

In [ ]:
# Colab auto-install (torch/sklearn are preinstalled; lightgbm is not):
try:
    import lightgbm
except ImportError:
    import subprocess, sys
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "lightgbm"], check=True)
    import lightgbm

import numpy as np
import lightgbm as lgb
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split

rng = np.random.default_rng(0)

# --- 0. Build the GOSS sampler by hand; recompute the lesson's worked example. ---
# 10 rows, one feature x, per-row gradients g.  a = b = 0.2.
x = np.array([1, 2, 3, 4, 5, 6, 7, 8, 9, 10], dtype=float)
g = np.array([0.9, -0.8, 0.05, -0.04, 0.03, -0.02, 0.01, -0.01, 0.7, -0.6])
a, b = 0.2, 0.2
fact = (1 - a) / b                       # Alg. 2 amplification weight
topN = int(a * len(g)); randN = int(b * len(g))
print("fact =", fact, " topN =", topN, " randN =", randN)   # fact = 4.0  topN = 2  randN = 2

order   = np.argsort(-np.abs(g))         # sort by |g| descending
topSet  = order[:topN]                   # set A: keep weight 1   -> rows x=1, x=2
# To match the worked example exactly we fix randSet = {x=9, x=5} (else: rng.choice(order[topN:], randN)):
randSet = np.array([8, 4])               # 0-based indices of x=9 and x=5  -> set B: weight = fact
w = np.ones(len(g)); w[randSet] = fact

# Score split at d=5 over A u B (Eqn. 1):
subset = np.concatenate([topSet, randSet])
left   = subset[x[subset] <= 5];  right = subset[x[subset] > 5]
sum_l  = float(np.sum(g[left]  * w[left]));  n_l = len(left)
sum_r  = float(np.sum(g[right] * w[right])); n_r = len(right)
V = (sum_l**2 / n_l + sum_r**2 / n_r) / len(g)
print(f"left sum = {sum_l:.2f} (n_l={n_l}), right sum = {sum_r:.2f} (n_r={n_r})")
print(f"GOSS variance gain V(d=5) = {V:.5f}")
# left sum = 0.22 (n_l=3), right sum = 2.80 (n_r=1)
# GOSS variance gain V(d=5) = 0.78561


# --- 1. Use the real LightGBM: histogram + leaf-wise GBDT, with the GOSS ablation. ---
X, y = make_classification(n_samples=20000, n_features=30, n_informative=12,
                           n_redundant=6, class_sep=0.8, random_state=0)
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.3, random_state=0)
dtrain = lgb.Dataset(Xtr, ytr); dtest = lgb.Dataset(Xte, yte, reference=dtrain)

common = dict(objective="binary", metric="binary_error", num_leaves=31,
              learning_rate=0.1, verbose=-1, seed=0)

def fit_and_score(boosting):
    params = {**common, "boosting": boosting}
    if boosting == "goss":
        params.update(top_rate=0.2, other_rate=0.2)   # a = 0.2, b = 0.2
    model = lgb.train(params, dtrain, num_boost_round=100)
    pred  = (model.predict(Xte) > 0.5).astype(int)
    return float((pred == yte).mean())

acc_full = fit_and_score("gbdt")   # standard: every tree uses all rows
acc_goss = fit_and_score("goss")   # GOSS: keep top 20%, sample 20% of the rest, re-weight
print(f"full-data GBDT test acc : {acc_full:.3f}")
print(f"GOSS GBDT      test acc : {acc_goss:.3f}   (each tree fit on ~{int((0.2+0.2)*len(ytr))} of {len(ytr)} rows)")
# GOSS matches full-data accuracy while fitting each tree on ~40% of the rows.
# (Exact numbers vary by version/seed; this is our small run, not the paper's reported result.)

## Visualize the data & results

_Does GOSS (keep large-gradient rows, sample + re-weight small-gradient ones) keep test accuracy while training each tree on far fewer rows?_

In [ ]:
# Reproduces the qualitative GOSS effect: ~same accuracy on far fewer rows per tree.
try:
    import lightgbm
except ImportError:
    import subprocess, sys
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "lightgbm"], check=True)
import numpy as np, lightgbm as lgb
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split

X, y = make_classification(n_samples=20000, n_features=30, n_informative=12,
                           n_redundant=6, class_sep=0.8, random_state=0)
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.3, random_state=0)
dtrain = lgb.Dataset(Xtr, ytr)
common = dict(objective="binary", num_leaves=31, learning_rate=0.1, verbose=-1, seed=0)

def acc(boosting):
    p = {**common, "boosting": boosting}
    if boosting == "goss": p.update(top_rate=0.2, other_rate=0.2)   # a=b=0.2
    m = lgb.train(p, dtrain, num_boost_round=100)
    return float(((m.predict(Xte) > 0.5).astype(int) == yte).mean())

a_full, a_goss = acc("gbdt"), acc("goss")
rows_full, rows_goss = len(ytr), int((0.2 + 0.2) * len(ytr))
print("full-data acc:", round(a_full, 3), " rows/tree:", rows_full)
print("GOSS acc     :", round(a_goss, 3), " rows/tree:", rows_goss)
# GOSS keeps accuracy within ~0.003 while each tree sees ~40% of the rows.

## Practice

Try each one in the empty cell below it, then reveal the worked solution.

**Problem 1.** The GOSS ablation. Train a gradient-boosting model on a toy dataset twice: once normally
            (uses all rows per tree) and once with GOSS (boosting='goss', keep top $a$, sample
            $b$, up-weight). What do you expect for test accuracy and for the number of rows each tree is fit
            on, and what does the comparison demonstrate?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Change only the boosting mode: keep the same trees, depth, learning rate, and data; flip GOSS on/off. — _An honest ablation isolates the one factor &mdash; gradient-based sampling &mdash; so any difference is attributable to it._
- Compare final test accuracy of the GOSS model vs the full-data model. — _GOSS keeps every large-gradient (poorly-fit) row and re-weights the sampled small-gradient rows, so the split statistics stay nearly unbiased &mdash; accuracy should hold._
- Note GOSS fits each tree on roughly $(a+b)\cdot N$ rows instead of $N$. — _Fewer rows scanned per split is exactly where the speed-up comes from; the $\frac{1-a}{b}$ weight is what keeps it accurate._

**Answer:** The GOSS model should reach about the same test accuracy while training each tree on
                 only ~$(a+b)\,N$ rows. Since the two models differ only by the sampling mode, this isolates
                 gradient-based one-side sampling as the cause of the speed-up. The kept large-gradient
                 rows carry the learning signal, and the $\frac{1-a}{b}$ re-weighting keeps the split score
                 unbiased &mdash; so we trade a small, controlled approximation error for far fewer rows per
                 tree. The CODEVIZ panel shows this with/without contrast on a toy run.

</details>

**Problem 2.** Recompute the worked example's split score from scratch. With $a=0.2$, $b=0.2$, on the subset
            $A\cup B$, the threshold $d=5$ gives left rows $(g{=}0.9,w{=}1),(g{=}-0.8,w{=}1),(g{=}0.03,w{=}4)$
            and right rows $(g{=}0.7,w{=}4)$, total $n=10$. Find fact, the weighted left/right
            gradient sums, and $\tilde V(d{=}5)$.

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- $\text{fact}=\frac{1-a}{b}=\frac{0.8}{0.2}=4$. — _The amplification weight on sampled small-gradient rows (Alg. 2)._
- Left weighted sum $=0.9-0.8+4(0.03)=0.22$ with $n_l=3$; right weighted sum $=4(0.7)=2.8$ with $n_r=1$. — _Each row's gradient times its weight, summed per side (the numerators inside Eqn. 1)._
- $\tilde V=\frac{1}{10}\!\left(\frac{0.22^2}{3}+\frac{2.8^2}{1}\right)=\frac{1}{10}(0.01613+7.84)=0.78561$. — _Plug the side sums into Eqn. (1): square, divide by side count, add, divide by $n$._

**Answer:** $\text{fact}=4$; left sum $=0.22$ ($n_l=3$), right sum $=2.8$ ($n_r=1$);
                 $\tilde V(d{=}5)=\frac{1}{10}\big(\frac{0.0484}{3}+7.84\big)=0.78561$. The notebook's first
                 cell recomputes these exact values, so a mismatch means a bug in the sampler or the weighting.

</details>

**Problem 3.** EFB merges feature A (values in $[0,10]$) and feature B (values in $[0,20]$), which are mutually
            exclusive, into one bundle. Using the paper's offset trick, what offset goes on B, what range does
            the bundle span, and how do you recover which original feature was active from a bundle value of
            $7$ versus $25$?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Offset B by the max of A, i.e. add $10$, so B now occupies $[10,30]$. — _Disjoint bin ranges let one stored value encode which feature was nonzero (Alg. 4)._
- The bundle spans $[0,30]$ (A's $[0,10]$ then B's offset $[10,30]$). — _A and B never fire together, so their ranges can sit side by side without collision._
- Value $7$ falls in $[0,10]$ &rarr; feature A was active (value $7$); value $25$ falls in $[10,30]$ &rarr; feature B was active (value $25-10=15$). — _The offset is reversible, so the merge is lossless &mdash; the histogram over the bundle equals the per-feature histograms combined._

**Answer:** Add offset $10$ to B (its range becomes $[10,30]$); the bundle spans $[0,30]$. A bundle value
                 of $7$ is in A's range, so feature A was active with value $7$; a value of $25$ is in B's
                 offset range, so feature B was active with value $25-10=15$. Because the offset is reversible,
                 bundling is lossless &mdash; that is why EFB cuts histogram cost from
                 $O(\#\text{data}\times\#\text{feature})$ to $O(\#\text{data}\times\#\text{bundle})$
                 without hurting accuracy.

</details>